# `02_prepare_data_for_analysis.ipynb`

This notebook is the **third step in the pipeline**. It merges AnnoTinder annotation exports,
Qualtrics survey/IAT data, and the X/Twitter dataset into analysis-ready datasets.

### Pipeline
`00_get-Xaccounts-MPs.ipynb` → `01_get_twitter_data.ipynb` → `02_prepare_data_for_analysis.ipynb`

### Inputs
- AnnoTinder exports (6 files):
  - `data/AnnoTinder_data_exports/annotations_95_AnnoBias job final, set 1.csv.csv`
  - `data/AnnoTinder_data_exports/annotations_96_AnnoBias job final, set 2.csv.csv`
  - `data/AnnoTinder_data_exports/annotations_97_AnnoBias job final, set 3.csv.csv`
  - `data/AnnoTinder_data_exports/annotations_98_AnnoBias job final, set 4.csv.csv`
  - `data/AnnoTinder_data_exports/annotations_99_AnnoBias job final, set 5.csv.csv`
  - `data/AnnoTinder_data_exports/annotations_100_AnnoBias job final, set 6.csv.csv`
- Qualtrics exports:
  - `data/Qualtrics_data_exports/start_survey.csv`
  - `data/Qualtrics_data_exports/before.csv`
  - `data/Qualtrics_data_exports/after.csv`
- Tweet dataset used for merging:
  - `data/X_data/final_stratified_immigration.csv`

### Outputs
The notebook writes three variants of the merged analysis dataset (each saved as `.csv`, `.parquet`, and `.pkl`):
- `data/final_merged_dataset_for_analysis.*`
- `data/final_merged_dataset_for_analysis_without_failing_attention_check.*`
- `data/final_merged_dataset_for_analysis_without_failing_attention_check_without_straightliners.*`


In [1]:
from datetime import datetime
import pandas as pd
import config
import rd_utils as rd
import io
import numpy as np

# ---------------------------
# Config: paths and filenames
# ---------------------------
PROJ = config.PROJECT_ROOT

AT_EXPORTS = f"{PROJ}/data/AnnoTinder_data_exports"
QUALTRICS  = f"{PROJ}/data/Qualtrics_data_exports"
X_DATA     = f"{PROJ}/data/X_data"

ANNOTINDER_FILES = [
    "annotations_95_AnnoBias job final, set 1.csv.csv",
    "annotations_96_AnnoBias job final, set 2.csv.csv",
    "annotations_97_AnnoBias job final, set 3.csv.csv",
    "annotations_98_AnnoBias job final, set 4.csv.csv",
    "annotations_99_AnnoBias job final, set 5.csv.csv",
    "annotations_100_AnnoBias job final, set 6.csv.csv",
]

QUALTRICS_FILES = {
    "start_survey": "start_survey.csv",
    "before": "before.csv",
    "after": "after.csv",
}


X_DATA_FILE = "final_stratified_immigration.csv"
CUTOFF_DATE = datetime(2024, 10, 14)

# ---------------------------
# Small helpers (non-I/O)
# ---------------------------
def load_qualtrics(name: str) -> pd.DataFrame:
    df = rd.read_csv(f"{QUALTRICS}/{QUALTRICS_FILES[name]}", skiprows=[1, 2])
    df["time_question"] = pd.to_datetime(df.get("StartDate"), errors="coerce")
    return df[df["time_question"] >= CUTOFF_DATE].copy()

def ensure_uid(df: pd.DataFrame, fallback_cols=("uid", "uid_before", "uid_after", "uid_x", "uid_y")) -> pd.DataFrame:
    if "uid" in df.columns:
        return df
    for c in fallback_cols:
        if c in df.columns:
            df = df.copy()
            df["uid"] = df[c]
            return df
    raise KeyError("Could not determine a 'uid' column (checked common fallback names).")

# ----------------------------------
# 1) Read AnnoTinder exports (6 files)
# ----------------------------------
annotinder_df = pd.concat(
    [
        rd.read_csv(f"{AT_EXPORTS}/{fname}").assign(condition_AT=i)
        for i, fname in enumerate(ANNOTINDER_FILES, start=1)
    ],
    ignore_index=True,
)

if "coder" not in annotinder_df.columns:
    raise KeyError("Expected column 'coder' not found in AnnoTinder export.")
annotinder_df["uid"] = annotinder_df["coder"]

# ----------------------------------
# 2) Read X data
# ----------------------------------
X_data = rd.read_csv(f"{X_DATA}/{X_DATA_FILE}")
print(f"Loaded X data: {len(X_data)} rows")

if "tweet_id" in X_data.columns:
    X_data["tweet_id"] = X_data["tweet_id"].astype(str)

# ----------------------------------
# 3) Read Qualtrics exports and filter on cutoff date
# ----------------------------------
start_survey = load_qualtrics("start_survey")
before       = load_qualtrics("before")
after        = load_qualtrics("after")

print(f"{len(start_survey)} participants started the experiment")
print(f"{len(before)} participants agreed to informed consent")
print(f"{len(after)} participants continued after IAT")

# ----------------------------------
# Merge Qualtrics datasets
# ----------------------------------
merge_key_ba = "uid" if ("uid" in before.columns and "uid" in after.columns) else "psid"
before_after = pd.merge(
    before, after, on=merge_key_ba, how="right", suffixes=("_before", "_after")
)
before_after = ensure_uid(before_after)

print(f"{len(before_after)} participants after merging before + after")

merge_key_ss = "psid" if ("psid" in start_survey.columns and "psid" in before_after.columns) else "uid"
qualtrics_merged = pd.merge(
    start_survey, before_after, on=merge_key_ss, how="inner"
)
qualtrics_merged = ensure_uid(qualtrics_merged)

print(f"{len(qualtrics_merged)} participants after merging start_survey with before/after")

# ----------------------------------
# Deduplication and filtering
# ----------------------------------
qualtrics_merged = qualtrics_merged.drop_duplicates(subset=["uid"], keep="first")
print(f"{len(qualtrics_merged)} participants after keeping first observation per uid")

qualtrics_merged = qualtrics_merged.dropna(subset=["dsc"])
print(f"{len(qualtrics_merged)} participants after removing missing IAT d-scores")


Loaded X data: 1860 rows
2338 participants started the experiment
2163 participants agreed to informed consent
1584 participants continued after IAT
1585 participants after merging before + after
1560 participants after merging start_survey with before/after
1543 participants after keeping first observation per uid
1443 participants after removing missing IAT d-scores


Merge AnnoTinder and Twitter data and Qualtrics data

In [2]:
# Define the cutoff date for filtering time_question
cutoff_date = datetime(2024, 10, 14)

# Step 1: Function to handle non-numeric values and trim the first 10 digits
def trim_unit_id(unit_id):
    try:
        # Try to convert to float, then to int, and slice the first 10 characters
        return str(int(float(unit_id)))[0:10]
    except ValueError:
        # Return the original value if it's non-numeric (handle as is)
        return unit_id

# Optionally, check the unique values to confirm '50PLUS' and 'BIJ1' are no longer present
print(f"Remaining parties in 'normalized_party': {X_data['normalized_party'].unique()}")

# Step 2: Apply the trim_unit_id function to 'unit_id' in annotinder_df and 'tweet_id' in X_data
annotinder_df['unit_id_15'] = annotinder_df['unit_id'].apply(trim_unit_id)
X_data['tweet_id_15'] = X_data['tweet_id'].apply(trim_unit_id)

# Step 3: Filter X_data to include only rows with matching tweet_id_15 in annotinder_df
matching_rows = X_data[X_data['tweet_id_15'].isin(annotinder_df['unit_id_15'])]

# Step 4: Merge all columns from X_data with annotinder_df based on tweet_id_15 and unit_id_15
annotinder_twitter = annotinder_df.merge(
    matching_rows, 
    left_on='unit_id_15', right_on='tweet_id_15', 
    how='left'
)

# Step 5: Ensure 'time_question' column is converted to datetime (if necessary)
if annotinder_twitter['time_question'].dtype != 'datetime64[ns]':
    annotinder_twitter['time_question'] = pd.to_datetime(
        annotinder_twitter['time_question'], 
        format="%Y-%m-%dT%H:%M:%S.%fZ", 
        errors='coerce'
    )

# Step 6: Filter annotinder_twitter to only include rows where time_question >= cutoff_date
annotinder_twitter = annotinder_twitter[annotinder_twitter['time_question'] >= cutoff_date]

# Step 7: Merge with qualtrics_merged DataFrame on 'uid'
if 'uid' in annotinder_twitter.columns and 'uid' in qualtrics_merged.columns:
    df = pd.merge(
        annotinder_twitter, 
        qualtrics_merged, 
        on='uid', 
        suffixes=('_annotinder', '_qualtrics')
    )
    print(f"After applying the cutoff date and merging, {len(df)} rows remain.")
else:
    print("Warning: 'uid' column not found in both DataFrames. Please check the column names.")


Remaining parties in 'normalized_party': ['D66' 'Volt' 'CDA' 'ChristenUnie' '50PLUS' 'Nieuw Sociaal Contract' 'SP'
 'GroenLinks' 'BIJ1' 'Partij voor de Dieren' 'PvdA' 'DENK'
 'GroenLinks-PvdA' 'PVV' 'Forum voor Democratie' 'JA21' 'VVD' 'SGP']
After applying the cutoff date and merging, 156346 rows remain.


In [3]:
print(f"for {df['uid'].nunique()} participants, the AnnoTinder, qualtrics and Twitter datasets could be merged")

for 1358 participants, the AnnoTinder, qualtrics and Twitter datasets could be merged


In [4]:
# Check and remove tweets from BIJ1 and 50PLUS as we do not have measurements for these parties:
print(f"Number of Annotations from BIJ1: {len(df[df['normalized_party'] == 'BIJ1'])}")
print(f"Number of Annotations from 50PLUS: {len(df[df['normalized_party'] == '50PLUS'])}")

Number of Annotations from BIJ1: 1632
Number of Annotations from 50PLUS: 1040


In [5]:
## Removing anntoations from BIj1 and 50plus:

# Remove rows from df where normalized_party is 'BIJ1' or '50PLUS'
print(f"Size of the dataset before removal: {len(df)}")

# Filter out rows with normalized_party as 'BIJ1' or '50PLUS'
df = df[~df['normalized_party'].isin(['BIJ1', '50PLUS'])]

print(f"Size of the dataset after removal: {len(df)}")

Size of the dataset before removal: 156346
Size of the dataset after removal: 153674


## ✦ Variable construction


In [6]:
df['uid'].nunique()

1358

In [7]:
pd.crosstab(df['condition_AT'], df['Condition'])

Condition,1,2,3,4,5,6
condition_AT,,,,,,
1,25303,0,0,0,0,0
2,0,27211,0,0,0,0
3,0,0,25150,0,0,0
4,0,0,0,25593,0,0
5,0,0,0,0,25100,0
6,0,0,0,0,0,25317


In [8]:
df['uid'].nunique()
df['uid'].nunique()
df[['uid', 'jobset']].nunique()

uid       1358
jobset      62
dtype: int64

In [9]:
print(df['normalized_party'].unique())

[nan 'ChristenUnie' 'D66' 'CDA' 'Volt' 'Nieuw Sociaal Contract' 'SP'
 'GroenLinks' 'Partij voor de Dieren' 'PvdA' 'DENK' 'GroenLinks-PvdA'
 'PVV' 'Forum voor Democratie' 'JA21' 'VVD' 'SGP']


In [10]:
# Sample likert_mapping (you have already defined this)
likert_mapping = {
    "Geheel onwaarschijnlijk": 1,
    "Onwaarschijnlijk": 2,
    "Niet waarschijnlijk, maar ook niet onwaarschijnlijk": 3,
    "Waarschijnlijk": 4,
    "Zeer waarschijnlijk": 5
}

# Assuming df is your DataFrame containing individual data

# Step 1: Map Likert responses to numeric values for each party
party_columns = ['PolCongr_1', 'PolCongr_2', 'PolCongr_3', 'PolCongr_4', 'PolCongr_5', 
                 'PolCongr_6', 'PolCongr_7', 'PolCongr_8', 'PolCongr_9', 'PolCongr_10',
                 'PolCongr_11', 'PolCongr_12', 'PolCongr_13', 'PolCongr_14', 'PolCongr_15']

# Map Likert scale responses to numeric values for all relevant columns
for party in party_columns:
    df[party] = df[party].map(likert_mapping)

# Step 2: Create a mapping from party names to their corresponding PolCongr_* columns
party_to_column_mapping = {
    'PVV': 'PolCongr_1',
    'GroenLinks-PvdA': 'PolCongr_2',  # GL-PvdA combined
    'VVD': 'PolCongr_3',
    'Nieuw Sociaal Contract': 'PolCongr_4',
    'D66': 'PolCongr_5',
    'BBB': 'PolCongr_6',
    'CDA': 'PolCongr_7',
    'SP': 'PolCongr_8',
    'DENK': 'PolCongr_9',
    'Partij voor de Dieren': 'PolCongr_10',
    'Forum voor Democratie': 'PolCongr_11',
    'SGP': 'PolCongr_12',
    'ChristenUnie': 'PolCongr_13',
    'Volt': 'PolCongr_14',
    'JA21': 'PolCongr_15'
}


# Step 3: Combine the relevant parties ('GroenLinks', 'PvdA', 'GroenLinks-PvdA') into a single group
df['normalized_party'] = df['normalized_party'].replace({
    'GroenLinks': 'GroenLinks-PvdA',  # Combine GroenLinks and PvdA into GroenLinks-PvdA
    'PvdA': 'GroenLinks-PvdA'
})


# Step 4: Calculate the score for the normalized party (single numeric score)
def calculate_party_score(row):
    normalized_party = row['normalized_party']
    
    # Look up the column corresponding to the normalized party
    normalized_party_column = party_to_column_mapping.get(normalized_party)
    
    if normalized_party_column:
        # Return the score from the corresponding column for the normalized party
        return row[normalized_party_column]
    else:
        return None

# Step 6: Apply the function to calculate the score for each participant
df['vote_likelihood_score'] = df.apply(calculate_party_score, axis=1)

# Step 7: Check the updated DataFrame with the vote likelihood score
#df[['normalized_party', 'vote_likelihood_score']].tail(50)
df['vote_likelihood_score'].value_counts()

df[['normalized_party', 'vote_likelihood_score']].tail(3)

,normalized_party,vote_likelihood_score
156343,VVD,1.0
156344,VVD,1.0
156345,VVD,1.0


In [11]:
df.groupby('normalized_party')['dsc'].mean().sort_values(ascending=False)

normalized_party
Partij voor de Dieren     0.548843
GroenLinks-PvdA           0.529799
DENK                      0.528117
SP                        0.514232
Forum voor Democratie     0.495250
ChristenUnie              0.491859
PVV                       0.491446
VVD                       0.490319
CDA                       0.488963
JA21                      0.483937
Volt                      0.476722
D66                       0.473886
Nieuw Sociaal Contract    0.467244
SGP                       0.457130
Name: dsc, dtype: float64

In [12]:
# Define the mapping for instructions
instruction_mapping = {
    1: "no instructions",
    2: "general instructions",
    3: "tailored instructions",
    4: "no instructions",
    5: "general instructions",
    6: "tailored instructions",
}

# Convert condition to integers before mapping
df["instruction_type"] = df["Condition"].astype(int).map(instruction_mapping)

# Define whether the condition includes author handle
df["source_shown"] = df["Condition"].astype(int).apply(lambda x: 0 if x in [1, 2, 3] else 1)

# Assuming df['Age_y'] contains the year of birth
current_year = datetime.now().year

# Create the 'Age' column by subtracting the year of birth from the current year
df['Age'] = 2024 - df['Age_qualtrics']
df['Male'] = df['gender'].map({'male': 1, 'female': 0, 'other/unknown': 0})

In [13]:
df['instruction_type'].value_counts()
df['source_shown'].value_counts()

source_shown
0    77664
1    76010
Name: count, dtype: int64

In [14]:
# Maak een nieuwe kolom 'Etniciteit_binair'
df['Etniciteit_binair'] = df['Etniciteit'].apply(lambda x: 1 if x == 'Nederlands' else 0)
df['Etniciteit_binair'].value_counts()

Etniciteit_binair
1    140523
0     13151
Name: count, dtype: int64

In [15]:
df['Werk'].value_counts()

Werk
Werkend (betaalde werknemer)                    91631
Niet werkend -- niet op zoek naar nieuw werk    28030
Anders                                          17743
Werkend (zelfstandig ondernemer)                 9568
Niet werkend -- op zoek naar werk                6448
Name: count, dtype: int64

In [16]:
df['Werk_binair'] = df['Werk'].apply(lambda x: 1 if x in ['Werkend (betaalde werknemer)', 'Werkend (zelfstandig ondernemer)'] else 0)
# Controleer de unieke waarden
print(df['Werk_binair'].value_counts())

Werk_binair
1    101199
0     52475
Name: count, dtype: int64


In [17]:
df['Attention1'].value_counts()

df['Attention_1_fail'] = df['Attention1'].apply(lambda x: 0 if x == 'De Volkskrant' else 1)
df['Attention_2_fail'] = df['Q68'].apply(lambda x: 0 if x == 'Altijd' else 1)

# Controleer de unieke waarden
df['Attention_1_fail'].value_counts()
df['Attention_2_fail'].value_counts()

# Maak een nieuwe kolom 'Both_Attention_Fail' die aangeeft of beide aandachtchecks zijn gemist
df['Both_Attention_Fail'] = ((df['Attention_1_fail'] == 1) & (df['Attention_2_fail'] == 1)).astype(int)

# Controleer de verdeling van de nieuwe kolom
print("\nVerdeling van de 'Both_Attention_Fail' kolom:")
print(df['Both_Attention_Fail'].value_counts())


Verdeling van de 'Both_Attention_Fail' kolom:
Both_Attention_Fail
0    151107
1      2567
Name: count, dtype: int64


In [18]:
df['Edu'].value_counts()

Edu
Hoger beroepsonderwijs (HBO, HTS, HEAO, Sociale Academie, HHNO, lerarenonderwijsetc.)                             48291
Middelbaar beroepsonderwijs (MBO, MTS, MEAO, Praktijkdiploma Boekhouden, Kleuterkweekschool, etc.)                44874
Wetenschappelijk onderwijs (universiteit)                                                                         19271
Voortgezet algemeen onderwijs (5-jaar HBS, MMS, HAVO, lyceum, atheneum, gymnasium, VWO, etc.)                     13964
Middelbaar algemeen onderwijs (LAVO, ULO, MULO, MAVO, 3-jaar HBS etc.)                                            12743
Lager beroepsonderwijs (LBO, LTS, LHNO, huishoud-/ambachts-school, LEAO, lager land-en tuinbouwonderwijs etc.)     7533
Voorbereidend of kort middelbaar beroepsonderwijs (VMBO, KMBO)                                                     4950
Lagere school (basisonderwijs)                                                                                     1024
Geen onderwijs gevolgd of het niet a

In [19]:
# Functie om de volledige indeling te maken
### Anders, nml is part of 'Basis of Voortgezet onderwijs' becuase the responses here indicated 'Hav' --> havo'''

def recode_edu(value):

    if value in [
        "Geen onderwijs gevolgd of het niet afgemaakt", 
        "Lagere school (basisonderwijs)"
    ]:
        return "Basisonderwijs"
    elif value in [
        "Voortgezet algemeen onderwijs (5-jaar HBS, MMS, HAVO, lyceum, atheneum, gymnasium, VWO, etc.)",
        "Anders, namelijk:"
    ]:
        return "Voortgezet onderwijs"
    elif value in [
        "Voorbereidend of kort middelbaar beroepsonderwijs (VMBO, KMBO)", 
        "Lager beroepsonderwijs (LBO, LTS, LHNO, huishoud-/ambachts-school, LEAO, lager land-en tuinbouwonderwijs etc.)", 
        "Middelbaar beroepsonderwijs (MBO, MTS, MEAO, Praktijkdiploma Boekhouden, Kleuterkweekschool, etc.)", 
        "Middelbaar algemeen onderwijs (LAVO, ULO, MULO, MAVO, 3-jaar HBS etc.)"
    ]:
        return "Praktijkopleiding"
    elif value in [
        "Hoger beroepsonderwijs (HBO, HTS, HEAO, Sociale Academie, HHNO, lerarenonderwijsetc.)", 
        "Wetenschappelijk onderwijs (universiteit)"
    ]:
        return "Hoger onderwijs"
    # Als er een onverwachte waarde is, raise een fout.
    else:
        raise ValueError(f"Onverwachte waarde: {value}")

# Nieuwe kolom toevoegen met de gerecodeerde waarden
df['Edu_breed'] = df['Edu'].apply(recode_edu)

# Controleer de resultaten
print(df['Edu_breed'].value_counts())

Edu_breed
Praktijkopleiding       70100
Hoger onderwijs         67562
Voortgezet onderwijs    14095
Basisonderwijs           1917
Name: count, dtype: int64


In [20]:
def recode_edu_numeric(value):
    """
    Recodes educational categories into 1 (High) and 0 (Rest).
    'Hoger onderwijs' is classified as '1' (High) and all other categories as '0' (Rest).
    """

    if value in [
        "Geen onderwijs gevolgd of het niet afgemaakt", 
        "Lagere school (basisonderwijs)",
        "Voortgezet algemeen onderwijs (5-jaar HBS, MMS, HAVO, lyceum, atheneum, gymnasium, VWO, etc.)",
        "Anders, namelijk:",
        "Voorbereidend of kort middelbaar beroepsonderwijs (VMBO, KMBO)", 
        "Lager beroepsonderwijs (LBO, LTS, LHNO, huishoud-/ambachts-school, LEAO, lager land-en tuinbouwonderwijs etc.)", 
        "Middelbaar beroepsonderwijs (MBO, MTS, MEAO, Praktijkdiploma Boekhouden, Kleuterkweekschool, etc.)", 
        "Middelbaar algemeen onderwijs (LAVO, ULO, MULO, MAVO, 3-jaar HBS etc.)"
    ]:
        return 0  # Group everything else as 'Rest' (0)
        
    elif value in [
        "Hoger beroepsonderwijs (HBO, HTS, HEAO, Sociale Academie, HHNO, lerarenonderwijsetc.)", 
        "Wetenschappelijk onderwijs (universiteit)"
    ]:
        return 1  # Group 'Hoger onderwijs' as 'High' (1)
    else:
        raise ValueError(f"Unexpected value: {value}")  # Raise error for unexpected values

# Recode the 'Edu_breed' column into numeric values
df['Edu_binary'] = df['Edu'].apply(recode_edu_numeric)

# Check the results
df['Edu_binary'].value_counts()

Edu_binary
0    86112
1    67562
Name: count, dtype: int64

In [21]:
df['PolOrient_1'].value_counts()

PolOrient_1
5.0     27698
7.0     24831
8.0     21828
6.0     20538
4.0     13182
2.0     11995
3.0     10884
9.0      8476
10.0     6111
1.0      5035
0.0      3096
Name: count, dtype: int64

In [22]:
# Functie om politieke oriëntatie te categoriseren
def recode_pol_orientation(value):
    if value <= 3:
        return "Links"
    elif value <= 6:
        return "Midden"
    elif value <= 10:
        return "Rechts"
    else:
        return "Onbekend"  # Dit zou je niet moeten tegenkomen, maar kan worden toegevoegd voor veiligheid.

# Nieuwe kolom toevoegen met de gerecodeerde waarden
df['PolOrientation_cat'] = df['PolOrient_1'].apply(recode_pol_orientation)

# Controleer de resultaten
print(df['PolOrientation_cat'].value_counts())

PolOrientation_cat
Midden    61418
Rechts    61246
Links     31010
Name: count, dtype: int64


In [23]:
df['AttitudeExtr1_1'].value_counts()

AttitudeExtr1_1
Niet eens, maar ook niet oneens    56693
Mee eens                           41607
Mee oneens                         29437
Zeer mee eens                      16849
Zeer mee oneens                     9088
Name: count, dtype: int64

In [24]:
# Functie om de antwoorden om te zetten naar numerieke waarden
def recode_attitude(value):
    if value == "Zeer mee eens":
        return 4
    elif value == "Mee eens":
        return 3
    elif value == "Niet eens, maar ook niet oneens":
        return 2
    elif value == "Mee oneens":
        return 1
    elif value == "Zeer mee oneens":
        return 0
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Functie voor omgekeerde codering (voor AttitudeExtr2_3)
def recode_attitude_reverse(value):
    if value == "Zeer mee eens":
        return 0
    elif value == "Mee eens":
        return 1
    elif value == "Niet eens, maar ook niet oneens":
        return 2
    elif value == "Mee oneens":
        return 3
    elif value == "Zeer mee oneens":
        return 4
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Pas de recodering toe op de vier kolommen, met omgekeerde codering voor AttitudeExtr2_3
df['AttitudeExtr2_1_numeric'] = df['AttitudeExtr2_1'].apply(recode_attitude)
df['AttitudeExtr2_2_numeric'] = df['AttitudeExtr2_2'].apply(recode_attitude)
df['AttitudeExtr2_3_numeric'] = df['AttitudeExtr2_3'].apply(recode_attitude_reverse)  # Omgekeerde codering
df['AttitudeExtr2_4_numeric'] = df['AttitudeExtr2_4'].apply(recode_attitude)

# Controleer op ontbrekende waarden
missing_values = df[['AttitudeExtr2_1_numeric', 'AttitudeExtr2_2_numeric', 'AttitudeExtr2_3_numeric', 'AttitudeExtr2_4_numeric']].isnull().sum()
print(f"Ontbrekende waarden per kolom:\n{missing_values}")

# Het samengestelde item maken (bijvoorbeeld het gemiddelde van de vier kolommen)
df['AttitudeExtr2_combined'] = df[['AttitudeExtr2_1_numeric', 'AttitudeExtr2_2_numeric', 'AttitudeExtr2_3_numeric', 'AttitudeExtr2_4_numeric']].mean(axis=1)

# Controleer de beschrijving van het nieuwe samengestelde item
print("\nSamengestelde score beschrijving:")
print(df['AttitudeExtr2_combined'].describe())

Ontbrekende waarden per kolom:
AttitudeExtr2_1_numeric    0
AttitudeExtr2_2_numeric    0
AttitudeExtr2_3_numeric    0
AttitudeExtr2_4_numeric    0
dtype: int64

Samengestelde score beschrijving:
count    153674.000000
mean          2.238524
std           0.934170
min           0.000000
25%           1.500000
50%           2.250000
75%           3.000000
max           4.000000
Name: AttitudeExtr2_combined, dtype: float64


In [25]:
# Functie om de antwoorden om te zetten met omgekeerde codering voor AttitudeExtr1_4
def recode_attitude_reverse(value):
    if value == "Zeer mee eens":
        return 0
    elif value == "Mee eens":
        return 1
    elif value == "Niet eens, maar ook niet oneens":
        return 2
    elif value == "Mee oneens":
        return 3
    elif value == "Zeer mee oneens":
        return 4
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Pas de recodering toe op de vier kolommen, met omgekeerde codering voor AttitudeExtr1_4
df['AttitudeExtr1_1_numeric'] = df['AttitudeExtr1_1'].apply(recode_attitude)
df['AttitudeExtr1_2_numeric'] = df['AttitudeExtr1_2'].apply(recode_attitude)
df['AttitudeExtr1_3_numeric'] = df['AttitudeExtr1_3'].apply(recode_attitude)
df['AttitudeExtr1_4_numeric'] = df['AttitudeExtr1_4'].apply(recode_attitude_reverse)  # Omgekeerde codering hier

# Controleer op ontbrekende waarden
missing_values = df[['AttitudeExtr1_1_numeric', 'AttitudeExtr1_2_numeric', 'AttitudeExtr1_3_numeric', 'AttitudeExtr1_4_numeric']].isnull().sum()
print(f"Ontbrekende waarden per kolom:\n{missing_values}")

# Het samengestelde item maken (bijvoorbeeld het gemiddelde van de vier kolommen)
df['AttitudeExtr1_combined'] = df[['AttitudeExtr1_1_numeric', 'AttitudeExtr1_2_numeric', 'AttitudeExtr1_3_numeric', 'AttitudeExtr1_4_numeric']].mean(axis=1)

# Controleer de beschrijving van het nieuwe samengestelde item
print("\nSamengestelde score beschrijving:")
print(df['AttitudeExtr1_combined'].describe())

Ontbrekende waarden per kolom:
AttitudeExtr1_1_numeric    0
AttitudeExtr1_2_numeric    0
AttitudeExtr1_3_numeric    0
AttitudeExtr1_4_numeric    0
dtype: int64

Samengestelde score beschrijving:
count    153674.000000
mean          2.311658
std           0.888020
min           0.000000
25%           1.750000
50%           2.250000
75%           3.000000
max           4.000000
Name: AttitudeExtr1_combined, dtype: float64


In [26]:
import pingouin as pg

# Cronbach's alpha berekenen voor de samengestelde schalen
# Zorg ervoor dat je de numerieke kolommen van de betreffende schalen gebruikt
alpha_1 = pg.cronbach_alpha(data=df[['AttitudeExtr1_1_numeric', 'AttitudeExtr1_2_numeric', 'AttitudeExtr1_3_numeric', 'AttitudeExtr1_4_numeric']])
alpha_2 = pg.cronbach_alpha(data=df[['AttitudeExtr2_1_numeric', 'AttitudeExtr2_2_numeric', 'AttitudeExtr2_3_numeric', 'AttitudeExtr2_4_numeric']])

# Print de resultaten van Cronbach's alpha
print(f"Cronbach's alpha voor AttitudeExtr1 schaal: {alpha_1[0]:.4f}")
print(f"Cronbach's alpha voor AttitudeExtr2 schaal: {alpha_2[0]:.4f}")

Cronbach's alpha voor AttitudeExtr1 schaal: 0.7860
Cronbach's alpha voor AttitudeExtr2 schaal: 0.8160


In [27]:
df[df['variable'] == 'stellingen.political_stance'][['Ideology', 'normalized_party','PolOrient_1']]

df['ImportIssue_1'].value_counts()

# Functie om de antwoorden om te zetten naar numerieke waarden voor ImportIssue_1
def recode_importance(value):
    if value == "Zeer belangrijk":
        return 4
    elif value == "Belangrijk":
        return 3
    elif value == "Niet belangrijk, maar ook niet onbelangrijk":
        return 2
    elif value == "Onbelangrijk":
        return 1
    elif value == "Zeer onbelangrijk":
        return 0
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Pas de recodering toe op de kolom ImportIssue_1
df['ImportIssue_1_numeric'] = df['ImportIssue_1'].apply(recode_importance)

# Controleer de beschrijving van de nieuwe numerieke schaal
print("\nSamenvatting van de nieuwe numerieke schaal:")
print(df['ImportIssue_1_numeric'].describe())

# Controleer de waardeverdeling
print("\nVerdeling van de nieuwe numerieke schaal:")
print(df['ImportIssue_1_numeric'].value_counts())


Samenvatting van de nieuwe numerieke schaal:
count    153674.000000
mean          3.015780
std           0.806659
min           0.000000
25%           3.000000
50%           3.000000
75%           4.000000
max           4.000000
Name: ImportIssue_1_numeric, dtype: float64

Verdeling van de nieuwe numerieke schaal:
ImportIssue_1_numeric
3    81152
4    41358
2    25497
1     3565
0     2102
Name: count, dtype: int64


In [28]:
df['KnowIssue_1'].value_counts()

KnowIssue_1
Niet veel, maar ook niet weinig    80226
Veel                               52066
Weinig                             10813
Heel veel                           8764
Heel weinig                         1805
Name: count, dtype: int64

In [29]:
# Functie om de antwoorden om te zetten naar numerieke waarden voor KnowIssue_1
def recode_knowledge(value):
    if value == "Heel veel":
        return 4
    elif value == "Veel":
        return 3
    elif value == "Niet veel, maar ook niet weinig":
        return 2
    elif value == "Weinig":
        return 1
    elif value == "Heel weinig":
        return 0
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Pas de recodering toe op de kolom KnowIssue_1
df['KnowIssue_1_numeric'] = df['KnowIssue_1'].apply(recode_knowledge)

# Controleer de beschrijving van de nieuwe numerieke schaal
print("\nSamenvatting van de nieuwe numerieke schaal:")
print(df['KnowIssue_1_numeric'].describe())

# Controleer de waardeverdeling
print("\nVerdeling van de nieuwe numerieke schaal:")
print(df['KnowIssue_1_numeric'].value_counts())


Samenvatting van de nieuwe numerieke schaal:
count    153674.000000
mean          2.359013
std           0.745242
min           0.000000
25%           2.000000
50%           2.000000
75%           3.000000
max           4.000000
Name: KnowIssue_1_numeric, dtype: float64

Verdeling van de nieuwe numerieke schaal:
KnowIssue_1_numeric
2    80226
3    52066
1    10813
4     8764
0     1805
Name: count, dtype: int64


In [30]:
df[df['variable'] == 'stellingen.political_stance'][['Ideology', 'normalized_party','PolOrient_1']]

df['instruction_type'].value_counts()
df['source_shown'].value_counts()

source_shown
0    77664
1    76010
Name: count, dtype: int64

In [31]:
df['variable'].value_counts()

variable
stellingen.misinformation      34811
stellingen.toxic               34809
stellingen.political_stance    34809
stellingen.sentiment           34808
confirm                         3980
political_stance_training       2642
sentiment_training              2618
toxic_training                  2603
misinformation_training         2594
Name: count, dtype: int64

In [32]:
# Encode the stance as an ordinal variable
scale_mapping = {
    "Helemaal niet waar": 1,
    "Gedeeltelijk niet waar": 2,
    "Neutraal": 3,
    "Gedeeltelijk waar": 4,
    "Helemaal waar": 5
}

df['value_nan'] = df['value'].replace(['Waar', 'Niet waar', 'confirmed'], np.nan)

# Map the remaining categories to a numeric scale
df['value_scaled'] = df['value_nan'].map(scale_mapping)
# Check the updated DataFrame
print(df[['value', 'value_scaled']].tail())

                     value  value_scaled
156341  Helemaal niet waar           1.0
156342            Neutraal           3.0
156343  Helemaal niet waar           1.0
156344  Helemaal niet waar           1.0
156345  Helemaal niet waar           1.0


In [33]:
# Check for missing values in key columns
df[df['variable'] == 'stellingen.toxic'][['vote_likelihood_score', 'source_shown', 'dsc', 'instruction_type', 'value_scaled']].isna().sum()

vote_likelihood_score    0
source_shown             0
dsc                      0
instruction_type         0
value_scaled             0
dtype: int64

In [34]:
df['variable'].value_counts()

variable
stellingen.misinformation      34811
stellingen.toxic               34809
stellingen.political_stance    34809
stellingen.sentiment           34808
confirm                         3980
political_stance_training       2642
sentiment_training              2618
toxic_training                  2603
misinformation_training         2594
Name: count, dtype: int64

## ✦ Saving the final merged dataset for analysis

In [35]:
PROJ = config.PROJECT_ROOT
base = "final_merged_dataset_for_analysis"

# CSV
rd.write_csv(df, f"{PROJ}/data/{base}.csv", quoting=1)

# Parquet
rd.write_parquet(df, f"{PROJ}/data/{base}.parquet")

# Pickle (if you added it; otherwise see below)
rd.write_pickle(df, f"{PROJ}/data/{base}.pkl")

In [36]:
## READ DATA BACK IN

In [37]:
df = rd.read_parquet(
    f"{PROJ}/data/final_merged_dataset_for_analysis.parquet"
)


In [38]:
# Define demographic fields you want to describe
demographic_fields = [
    'Age', 'Gender', 'Etniciteit_binair', 'Edu_breed',
    'PolOrientation_cat', 'dsc', 'vote_likelihood_score', 'instruction_type'
]

# Function to summarize active participants for one variable
def summarize_active_participants(df, variable_name, demographic_fields):
    # Filter to only rows for this variable
    active = df[df['variable'] == variable_name]

    # Collapse to one row per participant
    active_participants = active.groupby("uid")[demographic_fields].first()

    # Numbers
    n_total = df['uid'].nunique()
    n_active = active_participants.shape[0]

    print(f"\n=== {variable_name} ===")
    print(f"Total recruited: {n_total}")
    print(f"Active participants (≥1 annotation): {n_active}")

    # Continuous summary
    cont_vars = ['Age','dsc','vote_likelihood_score']
    cont_summary = active_participants[cont_vars].describe()
    print("\n--- Continuous Variables ---")
    print(cont_summary)

    # Categorical summary
    cat_vars = ['Gender','Etniciteit_binair','Edu_breed','PolOrientation_cat','instruction_type']
    print("\n--- Categorical Variables ---")
    for col in cat_vars:
        print(f"\n{col}")
        print(active_participants[col].value_counts(dropna=False))

# Run for all three stellingen
for var in ['stellingen.misinformation', 'stellingen.toxic', 'stellingen.sentiment']:
    summarize_active_participants(df, var, demographic_fields)


=== stellingen.misinformation ===
Total recruited: 1358
Active participants (≥1 annotation): 1248

--- Continuous Variables ---
               Age          dsc  vote_likelihood_score
count  1248.000000  1248.000000            1248.000000
mean     49.696314     0.506118               2.111378
std      15.615414     0.451057               1.230304
min      24.000000    -1.288011               1.000000
25%      36.000000     0.222671               1.000000
50%      50.000000     0.561151               2.000000
75%      63.000000     0.840145               3.000000
max      89.000000     1.528017               5.000000

--- Categorical Variables ---

Gender
Gender
Vrouw     655
Man       590
Anders      3
Name: count, dtype: int64

Etniciteit_binair
Etniciteit_binair
1    1144
0     104
Name: count, dtype: int64

Edu_breed
Edu_breed
Praktijkopleiding       574
Hoger onderwijs         545
Voortgezet onderwijs    114
Basisonderwijs           15
Name: count, dtype: int64

PolOrientation_cat


In [39]:
n_participants = df['uid'].nunique()

misinfo = df[df['variable'] == "stellingen.misinformation"]
len(set(misinfo['uid']))

# Basic demographic fields available
demographic_fields = [
    'Age', 'Gender', 'Etniciteit_binair', 'Edu_breed', 'PolOrientation_cat',
    'dsc', 'vote_likelihood_score', 'instruction_type'
]

# Check how many participants have non-null values for each key demographic variable
demographics_summary = df.groupby('uid')[demographic_fields].first().describe(include='all')

n_participants, demographics_summary.T[['count', 'mean', 'std', 'min', 'max']]

(1358,
                         count       mean        std       min       max
 Age                    1358.0  50.363034  15.718696      24.0      89.0
 Gender                   1358        NaN        NaN       NaN       NaN
 Etniciteit_binair      1358.0   0.918999   0.272938       0.0       1.0
 Edu_breed                1358        NaN        NaN       NaN       NaN
 PolOrientation_cat       1358        NaN        NaN       NaN       NaN
 dsc                    1358.0   0.515095   0.447804 -1.288011  1.528017
 vote_likelihood_score  1248.0   2.111378   1.230304       1.0       5.0
 instruction_type         1358        NaN        NaN       NaN       NaN)

## ✦ Creating samples with strickter QC settings for robustness checks 

In [40]:
print("Before filtering:", len(df))

# Remove rows where any attention check failed
df = df[
    (df['Attention_1_fail'] != True) &
    (df['Attention_2_fail'] != True) &
    (df['Both_Attention_Fail'] != True)
].copy()

# Print number of rows after filtering
print("After filtering:", len(df))

# Sanity check: confirm no failures remain
print("\nRemaining Attention_1_fail counts:")
print(df['Attention_1_fail'].value_counts())

print("\nRemaining Attention_2_fail counts:")
print(df['Attention_2_fail'].value_counts())

print("\nRemaining Both_Attention_Fail counts:")
print(df['Both_Attention_Fail'].value_counts())

Before filtering: 153674
After filtering: 143100

Remaining Attention_1_fail counts:
Attention_1_fail
0    143100
Name: count, dtype: int64

Remaining Attention_2_fail counts:
Attention_2_fail
0    143100
Name: count, dtype: int64

Remaining Both_Attention_Fail counts:
Both_Attention_Fail
0    143100
Name: count, dtype: int64


In [41]:
n_participants = df['uid'].nunique()

# Basic demographic fields available
demographic_fields = [
    'Age', 'Gender', 'Etniciteit_binair', 'Edu_breed', 'PolOrientation_cat',
    'dsc', 'vote_likelihood_score', 'instruction_type'
]

# Check how many participants have non-null values for each key demographic variable
demographics_summary = df.groupby('uid')[demographic_fields].first().describe(include='all')

n_participants, demographics_summary.T[['count', 'mean', 'std', 'min', 'max']]

(1270,
                         count      mean        std       min       max
 Age                    1270.0  51.06378  15.617922      24.0      89.0
 Gender                   1270       NaN        NaN       NaN       NaN
 Etniciteit_binair      1270.0  0.922047   0.268203       0.0       1.0
 Edu_breed                1270       NaN        NaN       NaN       NaN
 PolOrientation_cat       1270       NaN        NaN       NaN       NaN
 dsc                    1270.0  0.530691   0.441002 -1.192527  1.528017
 vote_likelihood_score  1163.0  2.066208   1.220134       1.0       5.0
 instruction_type         1270       NaN        NaN       NaN       NaN)

### ✦ Removing structural outliners 

In [42]:

# ==== CONFIG ====
ANNOTATOR_COL = "uid"                 # annotator id column
ANSWER_COL    = "value_scaled"        # numeric label column (1..5)

ANSWER_MAP = {  # leave empty if already numeric
    # "Strongly disagree": 1, "Disagree": 2, "Neutral": 3, "Agree": 4, "Strongly agree": 5
}

VARIABLES_OF_INTEREST = {
    "stellingen.misinformation",
    "stellingen.sentiment",
    "stellingen.toxic",
}

class QCConfig:
    # item-level timing
    min_rt_item_s: float = 0.8
    # annotator-level timing
    median_rt_floor_s: float = 1.0
    # straightlining ingredients
    longstring_k: int = 15
    sd_floor: float = 0.25
    entropy_floor_bits: float = 0.9
    # other optional flags
    edge_prop_thresh: float = 0.90
    repeat_tol: float = 1.0
    # calibration / honeypots
    cali_dev_tol: float = 1.0
    cali_miss_allow: int = 1
    cali_min_items: int = 3
    consensus_item_entropy_bits: float = 0.5

cfg = QCConfig()

# ==== HELPERS ====
def coerce_answer(series: pd.Series) -> pd.Series:
    if ANSWER_MAP:
        return series.map(ANSWER_MAP).astype("float")
    return pd.to_numeric(series, errors="coerce")

def shannon_entropy(counts) -> float:
    counts = np.array(counts, dtype=float)
    if counts.sum() == 0:
        return np.nan
    p = counts[counts > 0] / counts.sum()
    return float(-(p * np.log2(p)).sum())

def longest_run(values: pd.Series) -> int:
    vals = values.values
    best = cur = 0
    prev = object()
    for v in vals:
        if pd.isna(v):
            cur = 0
            prev = object()
            continue
        if v == prev:
            cur += 1
        else:
            cur = 1
        prev = v
        best = max(best, cur)
    return int(best)

def item_entropy(series: pd.Series) -> float:
    vc = series.value_counts(dropna=True)
    return shannon_entropy(vc.values)

def safe_quantile(s: pd.Series, q):
    try:
        return float(s.quantile(q))
    except Exception:
        return np.nan

# ==== PREP ====
def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    # keep variables of interest + optional training items
    mask = df["variable"].isin(VARIABLES_OF_INTEREST) | df["variable"].str.contains("training", case=False, na=False)
    df = df.loc[mask].copy()

    if "unit_status" in df.columns:
        df = df.loc[df["unit_status"].eq("DONE")].copy()

    # times and response time
    df["t_q"] = pd.to_datetime(df["time_question_annotinder"], utc=True, errors="coerce")
    df["t_a"] = pd.to_datetime(df["time_answer"], utc=True, errors="coerce")
    df["rt_s"] = (df["t_a"] - df["t_q"]).dt.total_seconds()
    df["rt_fast_flag"] = df["rt_s"] < cfg.min_rt_item_s

    # answers -> numeric
    df[ANSWER_COL] = coerce_answer(df[ANSWER_COL])

    # endpoints for edge proportion
    label_min = int(np.nanmin(df[ANSWER_COL].values))
    label_max = int(np.nanmax(df[ANSWER_COL].values))
    df["_label_min"] = label_min
    df["_label_max"] = label_max
    return df

# ==== CALIBRATION / HONEYPOT ====
def build_calibration_table(df: pd.DataFrame) -> pd.DataFrame:
    train_mask = df["variable"].str.contains("training", case=False, na=False)
    if train_mask.any():
        base = df.loc[train_mask].copy()
        src = "training"
    else:
        base = df.loc[df["variable"].isin(VARIABLES_OF_INTEREST)].copy()
        ent = (base.groupby(["variable", "unit_id"])[ANSWER_COL]
                    .apply(item_entropy)
                    .rename("entropy"))
        base = base.merge(ent, on=["variable", "unit_id"], how="left")
        base = base.loc[base["entropy"] <= cfg.consensus_item_entropy_bits].copy()
        src = "consensus-fallback"

    if base.empty:
        return pd.DataFrame(columns=["variable", "unit_id", "consensus", "n", "source"])

    def consensus_func(s):
        vc = s.value_counts()
        if vc.empty:
            return np.nan
        top = vc.max()
        modes = vc[vc == top].index.to_list()
        return float(modes[0]) if len(modes) == 1 else float(np.mean(modes))

    tab = (base.groupby(["variable", "unit_id"])
                .agg(consensus=(ANSWER_COL, consensus_func), n=(ANSWER_COL, "size"))
                .reset_index())
    tab["source"] = src
    return tab

# ==== REPEATED ITEMS ====
def per_annotator_repeat_inconsistency(df: pd.DataFrame) -> pd.Series:
    rep = df.groupby([ANNOTATOR_COL, "variable", "unit_id"]).size().rename("n").reset_index()
    rep = rep.loc[rep["n"] >= 2, [ANNOTATOR_COL, "variable", "unit_id"]]
    if rep.empty:
        return pd.Series(dtype=float, name="repeat_mad")

    merged = df.merge(rep, on=[ANNOTATOR_COL, "variable", "unit_id"], how="inner")

    def mad_pairs(s):
        v = s.dropna().values
        if v.size < 2:
            return np.nan
        diffs = [abs(v[i] - v[j]) for i in range(len(v)) for j in range(i + 1, len(v))]
        return float(np.median(diffs)) if diffs else np.nan

    out = (merged.groupby([ANNOTATOR_COL, "variable"])[ANSWER_COL]
                 .apply(mad_pairs)
                 .groupby(level=0).median()
                 .rename("repeat_mad"))
    return out

# ==== MAIN QC ====
def run_qc(df_raw: pd.DataFrame, cfg: QCConfig = cfg):
    df = prepare_df(df_raw)

    print(f"[QC] Input annotations after prep: {len(df)} rows")
    print(f"[QC] Unique annotators: {df[ANNOTATOR_COL].nunique()}")
    print(f"[QC] Variables: {sorted(df['variable'].dropna().unique().tolist())}")

    # per-annotator aggregates
    g_anno = df.groupby(ANNOTATOR_COL, as_index=False)
    speed = g_anno.agg(
        n_items=("unit_id", "size"),
        median_rt=("rt_s", "median"),
        p01_rt=("rt_s", lambda s: safe_quantile(s, 0.01)),
        p99_rt=("rt_s", lambda s: safe_quantile(s, 0.99)),
        fast_click_rate=("rt_fast_flag", "mean"),
    )
    speed["speed_flag"] = speed["median_rt"] < cfg.median_rt_floor_s

    label_min = int(df["_label_min"].iloc[0])
    label_max = int(df["_label_max"].iloc[0])

    def per_outcome_stats(d):
        d = d.sort_values("t_q")
        length = longest_run(d[ANSWER_COL])
        sd = float(d[ANSWER_COL].std(ddof=0))
        ent = item_entropy(d[ANSWER_COL])
        edge_prop = float(((d[ANSWER_COL] == label_min) | (d[ANSWER_COL] == label_max)).mean())
        return pd.Series({"longstring": length, "sd": sd, "entropy": ent, "edge_prop": edge_prop})

    stats = (df.groupby([ANNOTATOR_COL, "variable"])
               .apply(per_outcome_stats)
               .reset_index())

    worst = (stats.groupby(ANNOTATOR_COL)
                  .agg(longstring_max=("longstring", "max"),
                       sd_min=("sd", "min"),
                       entropy_min=("entropy", "min"),
                       edge_prop_max=("edge_prop", "max"))
                  .reset_index())

    worst["longstring_flag"] = worst["longstring_max"] >= cfg.longstring_k
    worst["lowvar_flag"]     = worst["sd_min"] < cfg.sd_floor
    worst["lowent_flag"]     = worst["entropy_min"] < cfg.entropy_floor_bits
    worst["edge_flag"]       = worst["edge_prop_max"] > cfg.edge_prop_thresh

    # structural straightliner: low entropy AND (longstring OR low variance)
    worst["struct_straightliner_flag"] = worst["lowent_flag"] & (worst["longstring_flag"] | worst["lowvar_flag"])

    # repeat consistency
    repeat = per_annotator_repeat_inconsistency(df).reset_index()
    if repeat.empty:
        repeat = pd.DataFrame({ANNOTATOR_COL: df[ANNOTATOR_COL].unique(), "repeat_mad": np.nan})
    repeat["repeat_flag"] = repeat["repeat_mad"] > cfg.repeat_tol

    # calibration/honeypots
    calitab = build_calibration_table(df)
    if not calitab.empty:
        joined = df.merge(calitab, on=["variable", "unit_id"], how="inner")
        joined["cali_miss"] = (joined[ANSWER_COL] - joined["consensus"]).abs() > cfg.cali_dev_tol
        cali = (joined.groupby(ANNOTATOR_COL)["cali_miss"]
                      .agg(["sum", "size"])
                      .rename(columns={"sum": "cali_misses", "size": "cali_n"})
                      .reset_index())
        cali["cali_flag"] = (cali["cali_n"] >= cfg.cali_min_items) & (cali["cali_misses"] > cfg.cali_miss_allow)
    else:
        cali = pd.DataFrame({
            ANNOTATOR_COL: df[ANNOTATOR_COL].unique(),
            "cali_misses": np.nan,
            "cali_n": 0,
            "cali_flag": False
        })

    # merge diagnostics
    qc = (speed.merge(worst, on=ANNOTATOR_COL, how="left")
                .merge(repeat, on=ANNOTATOR_COL, how="left")
                .merge(cali, on=ANNOTATOR_COL, how="left"))

    # guarded flag count
    base_flags = ["speed_flag", "longstring_flag", "lowvar_flag", "edge_flag", "repeat_flag", "cali_flag"]
    qc["n_flags"] = qc[base_flags].sum(axis=1, numeric_only=True) + qc["struct_straightliner_flag"].astype(int)

    def action_row(r):
        if (r["n_flags"] >= 3) or (r.get("cali_flag", False) and r.get("speed_flag", False)):
            return "drop"
        if r["n_flags"] >= 1:
            return "weight"
        return "keep"

    qc["action"] = qc.apply(action_row, axis=1)
    qc["weight"] = (1.0 - 0.2 * qc["n_flags"]).clip(lower=0.30, upper=1.00)

    # merge per-annotation
    df_qc = df.merge(qc[[ANNOTATOR_COL, "weight", "action", "struct_straightliner_flag"]],
                     on=ANNOTATOR_COL, how="left")

    DROP_UIDS = qc.loc[qc["action"] == "drop", ANNOTATOR_COL].dropna().sort_values().unique().tolist()
    DROP_STRUCT_UIDS = qc.query("action == 'drop' and struct_straightliner_flag")[ANNOTATOR_COL].dropna().sort_values().unique().tolist()
    ALL_STRUCT_UIDS = qc.loc[qc["struct_straightliner_flag"], ANNOTATOR_COL].dropna().sort_values().unique().tolist()

    # cleaned data for modeling (keeps weighted raters, removes 'drop')
    df_qc_clean = df_qc[df_qc["action"] != "drop"].copy()

    print(f"[QC] Annotators total: {qc.shape[0]}")
    print(f"[QC] Dropped annotators: {len(DROP_UIDS)}")
    print(f"[QC] Dropped structural straightliners: {len(DROP_STRUCT_UIDS)}")
    print(f"[QC] Kept annotations for modeling: {df_qc_clean.shape[0]} / {df_qc.shape[0]} "
          f"({df_qc_clean.shape[0]/max(1, df_qc.shape[0]):.1%})")

    return df_qc, qc, df_qc_clean, DROP_UIDS, DROP_STRUCT_UIDS, ALL_STRUCT_UIDS, stats, calitab


# === RUN ===
df_qc, qc_summary, df_qc_clean, DROP_UIDS, DROP_STRUCT_UIDS, ALL_STRUCT_UIDS, outcome_stats, calib_table = run_qc(df, cfg)

# ---------------------------
# SAVE "WITHOUT STRAIGHTLINERS" VERSION (i.e., excluding action == drop)
# ---------------------------

PROJ = config.PROJECT_ROOT
filename_base = "final_merged_dataset_for_analysis_without_failing_attention_check_without_straightliners"

# IMPORTANT: save the cleaned dataset, not the original df
to_save = df_qc_clean

rd.write_csv(to_save, f"{PROJ}/data/{filename_base}.csv", quoting=1)
rd.write_parquet(to_save, f"{PROJ}/data/{filename_base}.parquet")
rd.write_pickle(to_save, f"{PROJ}/data/{filename_base}.pkl")

print(f"[SAVE] Wrote: {PROJ}/data/{filename_base}.[csv|parquet|pkl]  (rows={len(to_save)})")

[QC] Input annotations after prep: 106898 rows
[QC] Unique annotators: 1232
[QC] Variables: ['misinformation_training', 'political_stance_training', 'sentiment_training', 'stellingen.misinformation', 'stellingen.sentiment', 'stellingen.toxic', 'toxic_training']


/tmp/ipykernel_1200265/3961952314.py:185: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(per_outcome_stats)


[QC] Annotators total: 1232
[QC] Dropped annotators: 141
[QC] Dropped structural straightliners: 141
[QC] Kept annotations for modeling: 94349 / 106898 (88.3%)
[SAVE] Wrote: ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/final_merged_dataset_for_analysis_without_failing_attention_check_without_straightliners.[csv|parquet|pkl]  (rows=94349)



How straightliners were removed?

```
 We identified structural straightliners at the annotator level using an order-aware and distribution-aware rule. For each outcome (misinformation, sentiment, toxicity) and annotator we computed: (a) the longest run of identical responses (“longstring”), (b) within-outcome response variance, and (c) response entropy. An annotator was flagged as a structural straightliner if they showed low entropy (≤ 0.9 bits) AND (longstring ≥ 15 identical responses in a row OR SD < 0.25).

Annotators were removed only when multiple QC indicators converged: action = drop if the total number of QC flags (including structural-straightliner, speed, longstring, low variance, edge-use, repeats, and calibration) was ≥ 3, or if both calibration and speed flags were triggered. All other annotators were retained and optionally down-weighted according to their flag count (weight = 1 − 0.2·flags, clipped to [0.30, 1.00]).

```
